# 🧠 Learning LLMs From Scratch — The macOS & Apple Silicon Way

**Machine:** MacBook Pro · M4 Pro · 48GB Unified Memory  
**Python:** 3.13 (.venv)  
**Philosophy:** Apple Silicon is the only platform that exists. MLX is our framework. Metal is our GPU API.

---

## Why Apple Silicon Changes Everything for ML

| Feature | What It Means for LLMs |
|---------|------------------------|
| **Unified Memory (48GB)** | CPU and GPU share the same RAM — no copying tensors between devices. A 30B 4-bit model fits entirely in memory. |
| **Metal GPU** | 16-core GPU on M4 Pro. Not CUDA, not OpenCL — Metal. Apple's own GPU compute API. |
| **Neural Engine** | 16-core dedicated ML accelerator (38 TOPS). Used by Core ML, not directly by MLX yet. |
| **Memory Bandwidth** | ~273 GB/s on M4 Pro. LLM inference is memory-bandwidth-bound, so this matters enormously. |
| **MLX** | Apple's open-source ML framework. NumPy-like API, lazy evaluation, runs natively on Metal. |
| **Metal Shading Language (MSL)** | C++-based language for writing GPU kernels. MLX uses this under the hood. |

---

## Roadmap

| Phase | Topic | Stack |
|-------|-------|-------|
| **0** | Environment & Apple Silicon Deep Dive | System tools, Metal |
| **1** | MLX Fundamentals — The NumPy of Apple Silicon | MLX |
| **2** | Math Foundations with MLX | MLX |
| **3** | Tokenization | SentencePiece, tiktoken |
| **4** | Embeddings & Positional Encoding | MLX |
| **5** | Self-Attention from Scratch | MLX |
| **6** | The Transformer — Block by Block | MLX |
| **7** | Building a GPT from Scratch | MLX |
| **8** | Training on Apple Silicon | MLX, unified memory |
| **9** | Modern Architectures — LLaMA, Mistral, Gemma | MLX |
| **10** | Metal Shading Language & Custom Kernels | MSL, Metal |
| **11** | Inference Optimization — Quantization, KV-Cache | MLX, mlx-lm |
| **12** | Serving Locally — mlx-lm, llama.cpp (Metal backend) | mlx-lm |
| **13** | Capstone — Run & Fine-tune Gemma 4 on Your Mac | MLX |

## Phase 0: Know Your Hardware

Before writing a single line of ML code, you need to understand what's under the hood. Apple Silicon is fundamentally different from the CPU+discrete-GPU architecture that most ML was built on.

### The Unified Memory Architecture (UMA)

```
Traditional (NVIDIA):
┌──────────┐    PCIe Bus    ┌──────────┐
│   CPU    │ ◄────────────► │   GPU    │
│  RAM     │   ~64 GB/s     │  VRAM    │
│  (64GB)  │   (bottleneck) │  (24GB)  │
└──────────┘                └──────────┘
  Data must be COPIED between CPU and GPU memory.
  GPU is limited to its own VRAM (e.g., 24GB on RTX 4090).

Apple Silicon (M4 Pro):
┌─────────────────────────────────────┐
│         Unified Memory (48GB)       │
│                                     │
│  ┌─────┐  ┌─────┐  ┌────────────┐  │
│  │ CPU │  │ GPU │  │ Neural Eng │  │
│  │ 14c │  │ 16c │  │    16c     │  │
│  └──┬──┘  └──┬──┘  └─────┬──────┘  │
│     │        │            │         │
│     └────────┴────────────┘         │
│         All share the SAME          │
│         memory pool @ 273 GB/s      │
└─────────────────────────────────────┘
  NO copying. CPU tensor IS the GPU tensor.
  48GB available to ALL processors.
```

### Why this matters for LLMs:
- **No VRAM limit**: A 7B float16 model needs ~14GB. On NVIDIA, you need a GPU with ≥14GB VRAM. On your Mac, it just uses unified memory.
- **No transfer overhead**: Moving a 14GB model to GPU on PCIe takes ~220ms. On Apple Silicon: 0ms.
- **LLM inference is memory-bandwidth-bound**: Generating each token requires reading the entire model weights. At 273 GB/s, your M4 Pro reads a 7B fp16 model in ~51ms per token (~20 tok/s).

---
# ═══════════════════════════════════════════════════════════════
# PHASE 0: Your Machine — Understanding Apple Silicon
# ═══════════════════════════════════════════════════════════════

In [ ]:
# ============================================================
# 0.1 Interrogate Your Machine
# ============================================================
import subprocess
import platform
import os

print("=" * 60)
print("SYSTEM INFORMATION")
print("=" * 60)

# macOS version
print(f"macOS version:  {platform.mac_ver()[0]}")
print(f"Architecture:   {platform.machine()}")
print(f"Python:         {platform.python_version()}")

# Chip info via sysctl
def sysctl(key):
    try:
        result = subprocess.run(["sysctl", "-n", key], capture_output=True, text=True)
        return result.stdout.strip()
    except:
        return "N/A"

print(f"\nChip:           {sysctl('machdep.cpu.brand_string')}")
print(f"CPU cores:      {sysctl('hw.perflevel0.logicalcpu')} performance + {sysctl('hw.perflevel1.logicalcpu')} efficiency")
print(f"Total memory:   {int(sysctl('hw.memsize')) / (1024**3):.0f} GB")

# GPU info via system_profiler
gpu_info = subprocess.run(
    ["system_profiler", "SPDisplaysDataType"],
    capture_output=True, text=True
).stdout
for line in gpu_info.split("\n"):
    line = line.strip()
    if any(k in line for k in ["Chipset", "Cores", "Metal", "Total Number of Cores"]):
        print(f"GPU:            {line}")

In [ ]:
# ============================================================
# 0.4 Install Everything We Need (run once)
# ============================================================
# Uncomment and run this cell to install all dependencies.
# We're using your .venv with Python 3.13.

# !pip install --upgrade pip
# !pip install mlx mlx-lm mlx-nn
# !pip install numpy matplotlib seaborn
# !pip install tiktoken sentencepiece
# !pip install huggingface_hub transformers datasets
# !pip install llama-cpp-python  # Builds with Metal support on macOS automatically
# !pip install einops

# Verify MLX installation
import mlx.core as mx
import mlx.nn as nn
import numpy as np

print(f"MLX version:     {mx.__version__}" if hasattr(mx, '__version__') else "MLX loaded")
print(f"Default device:  {mx.default_device()}")
print(f"NumPy version:   {np.__version__}")
print()
print("Available MLX backends:")
print(f"  GPU (Metal):   {mx.metal.is_available()}")
print(f"  Metal device:  {mx.default_device()}")

# Quick sanity check
a = mx.random.normal((1000, 1000))
b = mx.random.normal((1000, 1000))
c = a @ b
mx.eval(c)  # MLX is lazy — must explicitly evaluate
print(f"\n✅ MLX matrix multiply works. Result shape: {c.shape}")

In [ ]:
# ============================================================
# 0.2 Metal GPU Capabilities
# ============================================================
# Metal is Apple's GPU API (like CUDA for NVIDIA, but for Apple GPUs).
# Let's see what Metal features your GPU supports.

import subprocess
import json

# Check Metal support
result = subprocess.run(
    ["system_profiler", "SPDisplaysDataType", "-json"],
    capture_output=True, text=True
)

try:
    data = json.loads(result.stdout)
    displays = data.get("SPDisplaysDataType", [{}])
    for gpu in displays:
        print("Metal GPU Information")
        print("=" * 40)
        for key, value in gpu.items():
            if key != "spdisplays_ndrvs":  # Skip display info
                print(f"  {key}: {value}")
except json.JSONDecodeError:
    print("Could not parse GPU info as JSON. Raw output:")
    print(result.stdout[:500])

print("\n" + "=" * 40)
print("Metal Feature Sets on Apple Silicon:")
print("=" * 40)
print("""
• Metal 3          — Latest API (M3/M4 family)
• GPU Family Apple 9 — M4 Pro GPU family
• Tile Shading     — On-chip tile memory for efficient compute
• Mesh Shaders     — Advanced geometry processing
• Ray Tracing      — Hardware-accelerated (not relevant for ML)
• Shared Memory    — Threadgroup shared memory for fast inter-thread communication

For ML, the key Metal features are:
• Compute Shaders  — General-purpose GPU compute (what MLX uses)
• Shared Memory    — Fast scratchpad for matrix multiply tiles
• SIMD Groups      — 32 threads executing in lockstep (like CUDA warps)
• Threadgroups     — Groups of SIMD groups (like CUDA thread blocks)
""")

In [ ]:
# ============================================================
# 0.3 Memory Bandwidth — The Key Metric for LLM Inference
# ============================================================
# LLM token generation is MEMORY-BANDWIDTH BOUND, not compute-bound.
# Each token requires reading ALL model weights from memory once.
# So: tokens/sec ≈ memory_bandwidth / model_size

print("Why Memory Bandwidth Matters for LLMs")
print("=" * 55)
print()

# M4 Pro specs
bandwidth_gbs = 273  # GB/s for M4 Pro
memory_gb = 48

# Model sizes (approximate, weights only)
models = {
    "GPT-2 (124M, fp16)":       0.25,
    "LLaMA 3 7B (fp16)":        14.0,
    "LLaMA 3 7B (4-bit quant)": 3.5,
    "Mistral 7B (4-bit)":       3.5,
    "LLaMA 3 13B (4-bit)":      6.5,
    "Gemma 4 12B (4-bit)":      6.0,
    "LLaMA 3 70B (4-bit)":      35.0,
    "Gemma 4 27B (4-bit)":      13.5,
}

print(f"Your M4 Pro: {bandwidth_gbs} GB/s bandwidth, {memory_gb} GB memory")
print()
print(f"{'Model':<30} {'Size':>8} {'Fits?':>6} {'~tok/s':>8} {'~ms/tok':>8}")
print("-" * 65)

for name, size_gb in models.items():
    fits = "✅" if size_gb < memory_gb * 0.85 else "⚠️"  # Leave room for KV-cache
    # Theoretical max: bandwidth / model_size
    # Real-world is ~60-70% of theoretical due to overhead
    theoretical_tps = bandwidth_gbs / size_gb
    practical_tps = theoretical_tps * 0.65  # Conservative estimate
    ms_per_tok = 1000 / practical_tps
    print(f"{name:<30} {size_gb:>6.1f}GB {fits:>5} {practical_tps:>7.1f} {ms_per_tok:>7.1f}")

print()
print("💡 Key takeaway: With 4-bit quantization, you can run up to ~35B models.")
print("   The 70B model barely fits but leaves no room for KV-cache.")
print("   Sweet spot for your machine: 7B–27B models with 4-bit quantization.")

---
# ═══════════════════════════════════════════════════════════════
# PHASE 1: MLX Fundamentals — Your ML Framework
# ═══════════════════════════════════════════════════════════════

In [ ]:
# ============================================================
# 1.5 Training Loop Pattern in MLX
# ============================================================
# MLX uses a functional pattern for training.
# Key difference from PyTorch: no .backward(), no optimizer.step()
# Instead: mx.value_and_grad() + optimizer.update()

import mlx.core as mx
import mlx.nn as nn
import mlx.optimizers as optim

# Simple model: 2-layer MLP
class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 32)
        self.fc2 = nn.Linear(32, 1)
    
    def __call__(self, x):
        x = nn.relu(self.fc1(x))
        return self.fc2(x)

# Create model and optimizer
model = TinyMLP()
optimizer = optim.Adam(learning_rate=1e-3)

# The MLX training pattern:
# 1. Define a loss function that takes (model, x, y)
# 2. Use nn.value_and_grad() to get both loss AND gradients
# 3. Use optimizer.update() to apply gradients

def loss_fn(model, x, y):
    pred = model(x)
    return mx.mean((pred - y) ** 2)

# This creates a function that returns (loss_value, gradients)
loss_and_grad_fn = nn.value_and_grad(model, loss_fn)

# Generate some dummy data (y = sum of inputs)
np.random.seed(42)
X = mx.array(np.random.randn(100, 4).astype(np.float32))
y = X.sum(axis=-1, keepdims=True)  # Target: sum of features

# Training loop
print("Training a tiny MLP on y = sum(x):")
print("-" * 40)
for epoch in range(200):
    loss, grads = loss_and_grad_fn(model, X, y)
    optimizer.update(model, grads)
    mx.eval(model.parameters(), optimizer.state)  # Evaluate lazily
    
    if epoch % 40 == 0:
        print(f"  Epoch {epoch:>3d}: loss = {loss.item():.6f}")

# Test
test_x = mx.array([[1.0, 2.0, 3.0, 4.0]])
pred = model(test_x)
mx.eval(pred)
print(f"\nTest: input={test_x.tolist()[0]}, sum={sum([1,2,3,4])}, predicted={pred.item():.3f}")

## Phase 1: MLX — Apple's ML Framework

### What is MLX?
MLX is Apple's open-source ML framework, built specifically for Apple Silicon. Think of it as:
- **NumPy** for the API feel (familiar, Pythonic)
- **PyTorch** for the autograd and neural network modules
- **JAX** for the lazy evaluation and functional transforms

### Key Design Principles:
1. **Lazy Evaluation**: Operations are recorded but not executed until you call `mx.eval()`. This allows MLX to fuse operations and optimize the compute graph.
2. **Unified Memory**: Arrays live in shared memory. No `.to(device)` calls. CPU and GPU see the same data.
3. **Composable Transforms**: `mx.grad()`, `mx.vmap()`, `mx.compile()` — functional transforms like JAX.
4. **Metal Backend**: All GPU compute goes through Metal compute shaders.

### MLX vs PyTorch on Mac:
| Feature | PyTorch (MPS) | MLX |
|---------|--------------|-----|
| GPU backend | MPS (Metal Performance Shaders) | Metal compute shaders |
| Memory model | Copies to MPS device | True unified, zero-copy |
| Evaluation | Eager (immediate) | Lazy (deferred) |
| Maturity | Very mature, huge ecosystem | Younger, Apple-focused |
| Performance on Apple Silicon | Good | Better (native) |
| Autograd | Yes | Yes |
| Custom kernels | Not easy on Metal | Metal shading language |

In [ ]:
# ============================================================
# 1.4 MLX Neural Network Modules (mlx.nn)
# ============================================================
# mlx.nn provides building blocks similar to torch.nn

import mlx.nn as nn
import mlx.core as mx

# Linear layer (the most fundamental building block)
linear = nn.Linear(input_dims=8, output_dims=16)

# Inspect the parameters
print("Linear layer parameters:")
for name, param in linear.parameters().items():
    print(f"  {name}: shape={param.shape}, dtype={param.dtype}")

# Forward pass
x = mx.random.normal((2, 4, 8))  # (batch=2, seq_len=4, features=8)
y = linear(x)
mx.eval(y)
print(f"\nInput:  {x.shape}")
print(f"Output: {y.shape}")

print("\n" + "=" * 40)
print("Key mlx.nn modules for building Transformers:")
print("=" * 40)

modules = {
    "nn.Linear":        "Dense/fully-connected layer (y = xW^T + b)",
    "nn.Embedding":     "Token embedding lookup table",
    "nn.LayerNorm":     "Layer normalization (stabilizes training)",
    "nn.RMSNorm":       "RMS normalization (used in LLaMA, Gemma)",
    "nn.MultiHeadAttention": "Multi-head self-attention",
    "nn.ReLU / nn.GELU / nn.SiLU": "Activation functions",
    "nn.Dropout":       "Regularization (randomly zero out values)",
}

for module, desc in modules.items():
    print(f"  {module:<30} {desc}")

In [ ]:
# ============================================================
# 1.1 MLX Arrays — The Fundamental Data Type
# ============================================================
import mlx.core as mx
import numpy as np

# Creating arrays (just like NumPy)
a = mx.array([1.0, 2.0, 3.0, 4.0])
print(f"1D array: {a}")
print(f"  dtype: {a.dtype}")
print(f"  shape: {a.shape}")

# 2D array (matrix)
M = mx.array([[1, 2, 3],
              [4, 5, 6]])
print(f"\n2D array:\n{M}")
print(f"  shape: {M.shape}")

# From NumPy (zero-copy when possible!)
np_arr = np.random.randn(3, 4).astype(np.float32)
mx_arr = mx.array(np_arr)
print(f"\nFrom NumPy: shape={mx_arr.shape}, dtype={mx_arr.dtype}")

# Back to NumPy
back_to_np = np.array(mx_arr)
print(f"Back to NumPy: {back_to_np.shape}")

# Key dtypes for LLMs
print("\nKey dtypes for LLM work:")
for dtype in [mx.float32, mx.float16, mx.bfloat16]:
    x = mx.ones((2, 2), dtype=dtype)
    print(f"  {dtype}: {x.nbytes} bytes for 2x2 matrix")

In [ ]:
# ============================================================
# 1.3 Automatic Differentiation (Autograd) in MLX
# ============================================================
# Training neural networks requires gradients.
# MLX uses functional transforms (like JAX), not tape-based (like PyTorch).

import mlx.core as mx

# Simple function: f(x) = x^2
def f(x):
    return x ** 2

# Get the gradient function
df = mx.grad(f)

# Evaluate at x = 3.0
x = mx.array(3.0)
print(f"f(3.0)  = {f(x).item()}")       # 9.0
print(f"f'(3.0) = {df(x).item()}")      # 6.0 (derivative of x^2 is 2x)

# Second derivative
ddf = mx.grad(mx.grad(f))
print(f"f''(3.0) = {ddf(x).item()}")    # 2.0 (second derivative of x^2 is 2)

print("\n" + "=" * 40)

# More realistic: gradient of a loss function
def mse_loss(predictions, targets):
    return mx.mean((predictions - targets) ** 2)

# mx.grad takes gradient w.r.t. the FIRST argument by default
loss_grad_fn = mx.grad(mse_loss)

preds = mx.array([1.0, 2.0, 3.0])
targets = mx.array([1.5, 2.5, 3.5])

loss = mse_loss(preds, targets)
grads = loss_grad_fn(preds, targets)

print(f"Predictions: {preds}")
print(f"Targets:     {targets}")
print(f"MSE Loss:    {loss.item():.4f}")
print(f"Gradients:   {grads}")
print("\n💡 Gradients point in the direction that INCREASES loss.")
print("   We subtract them during training (gradient descent).")

In [ ]:
# ============================================================
# 1.2 Lazy Evaluation — MLX's Superpower
# ============================================================
# This is the BIGGEST difference from PyTorch/NumPy.
# Operations are NOT executed immediately. They build a compute graph.
# Only mx.eval() (or printing/converting) triggers actual computation.

import time

# These operations are RECORDED, not executed
a = mx.random.normal((5000, 5000))
b = mx.random.normal((5000, 5000))
c = a @ b          # Matrix multiply — NOT computed yet!
d = mx.tanh(c)     # Activation — NOT computed yet!
e = d.sum()        # Reduction — NOT computed yet!

print("Operations recorded (not yet computed).")
print(f"e = {e}")
print("  ↑ Notice: MLX shows the value because printing triggers eval.")
print()

# Why lazy evaluation matters:
# MLX can FUSE operations. Instead of:
#   1. Compute c = a @ b  (write 5000x5000 to memory)
#   2. Read c, compute d = tanh(c)  (write 5000x5000 to memory)
#   3. Read d, compute e = sum(d)
# MLX can do:
#   1. Compute a @ b, apply tanh, accumulate sum — all in ONE pass
# This saves memory bandwidth, which is the bottleneck on Apple Silicon.

# Benchmark: lazy vs forced-eager
def benchmark_lazy():
    a = mx.random.normal((2000, 2000))
    b = mx.random.normal((2000, 2000))
    c = a @ b
    d = mx.relu(c)
    e = d.sum()
    mx.eval(e)  # Single eval at the end
    return e

def benchmark_eager():
    a = mx.random.normal((2000, 2000))
    mx.eval(a)  # Force eval
    b = mx.random.normal((2000, 2000))
    mx.eval(b)  # Force eval
    c = a @ b
    mx.eval(c)  # Force eval
    d = mx.relu(c)
    mx.eval(d)  # Force eval
    e = d.sum()
    mx.eval(e)  # Force eval
    return e

# Warmup
benchmark_lazy()
benchmark_eager()

# Time them
start = time.perf_counter()
for _ in range(10):
    benchmark_lazy()
lazy_time = (time.perf_counter() - start) / 10

start = time.perf_counter()
for _ in range(10):
    benchmark_eager()
eager_time = (time.perf_counter() - start) / 10

print(f"Lazy evaluation:  {lazy_time*1000:.1f} ms")
print(f"Forced eager:     {eager_time*1000:.1f} ms")
print(f"Speedup:          {eager_time/lazy_time:.2f}x")
print("\n💡 Lazy eval lets MLX fuse operations and reduce memory traffic.")

---
# ═══════════════════════════════════════════════════════════════
# PHASE 2: The Math — Implemented in MLX
# ═══════════════════════════════════════════════════════════════

In [ ]:
# ============================================================
# 2.1.4 Temperature & Sampling — Controlling Generation
# ============================================================
import mlx.core as mx
import matplotlib.pyplot as plt
import numpy as np

logits = mx.array([2.0, 1.0, 0.1, -1.0, 3.0])
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]

fig, axes = plt.subplots(1, len(temperatures), figsize=(20, 4))
for i, temp in enumerate(temperatures):
    probs = mx.softmax(logits / temp, axis=-1)
    mx.eval(probs)
    probs_np = np.array(probs.tolist())
    axes[i].bar(range(5), probs_np, color='#009688')
    axes[i].set_title(f'T = {temp}')
    axes[i].set_ylim(0, 1)
    axes[i].set_xlabel('Token')
    if i == 0:
        axes[i].set_ylabel('Probability')

plt.suptitle('Effect of Temperature on Token Probabilities', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("T=0.1 → Nearly deterministic (always picks the top token)")
print("T=1.0 → Standard softmax")
print("T=5.0 → Nearly uniform (very random, often incoherent)")

# Top-k and Top-p (nucleus) sampling
print("\n" + "=" * 50)
print("Sampling Strategies")
print("=" * 50)

def top_k_sample(logits, k=3, temperature=1.0):
    """Keep only top-k logits, zero out the rest."""
    scaled = logits / temperature
    # Get top-k indices
    top_k_vals = mx.sort(scaled)[-k:]  # k largest values
    threshold = top_k_vals[0]  # Smallest of top-k
    # Mask out everything below threshold
    masked = mx.where(scaled >= threshold, scaled, mx.array(float('-inf')))
    probs = mx.softmax(masked, axis=-1)
    return probs

def top_p_sample(logits, p=0.9, temperature=1.0):
    """Nucleus sampling: keep smallest set of tokens with cumulative prob >= p."""
    scaled = logits / temperature
    probs = mx.softmax(scaled, axis=-1)
    mx.eval(probs)
    
    # Sort descending
    sorted_indices = mx.argsort(-probs)
    sorted_probs = probs[sorted_indices]
    mx.eval(sorted_probs)
    
    # Cumulative sum
    cumsum = mx.cumsum(sorted_probs, axis=-1)
    mx.eval(cumsum)
    
    # Find cutoff
    cutoff_idx = 0
    cumsum_list = cumsum.tolist()
    for idx, cs in enumerate(cumsum_list):
        if cs >= p:
            cutoff_idx = idx
            break
    
    print(f"  Top-p={p}: keeping {cutoff_idx + 1} tokens "
          f"(cumulative prob = {cumsum_list[cutoff_idx]:.3f})")
    return probs

print("\nTop-k sampling (k=3):")
top_k_probs = top_k_sample(logits, k=3)
mx.eval(top_k_probs)
print(f"  Probs: {[f'{p:.4f}' for p in top_k_probs.tolist()]}")

print("\nTop-p (nucleus) sampling (p=0.9):")
top_p_probs = top_p_sample(logits, p=0.9)

## 2.1 Essential Math for LLMs (All in MLX)

Everything here runs on your M4 Pro's Metal GPU. No CUDA, no CPU fallback.

In [ ]:
# ============================================================
# 2.1.3 Softmax — Turning Scores into Probabilities
# ============================================================
import mlx.core as mx
import matplotlib.pyplot as plt
import numpy as np

# Softmax is used everywhere in LLMs:
#   1. Attention weights (which tokens to focus on)
#   2. Output layer (which token to generate next)

def softmax_manual(x):
    """Manual softmax with numerical stability trick."""
    x_shifted = x - mx.max(x, axis=-1, keepdims=True)  # Prevent overflow
    exp_x = mx.exp(x_shifted)
    return exp_x / mx.sum(exp_x, axis=-1, keepdims=True)

logits = mx.array([2.0, 1.0, 0.1, -1.0, 3.0])

probs_manual = softmax_manual(logits)
probs_builtin = mx.softmax(logits, axis=-1)

mx.eval(probs_manual, probs_builtin)

print(f"Logits:           {logits.tolist()}")
print(f"Softmax (manual): {[f'{p:.4f}' for p in probs_manual.tolist()]}")
print(f"Softmax (MLX):    {[f'{p:.4f}' for p in probs_builtin.tolist()]}")
print(f"Sum:              {mx.sum(probs_builtin).item():.6f} (always 1.0)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
logits_np = np.array(logits.tolist())
probs_np = np.array(probs_builtin.tolist())

axes[0].bar(range(5), logits_np, color='#2196F3')
axes[0].set_title('Raw Logits (before softmax)')
axes[0].set_ylabel('Score')
axes[1].bar(range(5), probs_np, color='#FF5722')
axes[1].set_title('Probabilities (after softmax)')
axes[1].set_ylabel('Probability')
for ax in axes:
    ax.set_xlabel('Token index')
plt.tight_layout()
plt.show()

print("\n💡 Softmax amplifies differences. Token 4 (logit=3.0) gets ~56% probability.")

In [ ]:
# ============================================================
# 2.1.1 Dot Product — The Foundation of Attention
# ============================================================
import mlx.core as mx
import numpy as np

# The dot product measures similarity between two vectors.
# In attention: "how much should token A attend to token B?"

query = mx.array([1.0, 0.5, -0.3, 0.8])   # "What am I looking for?"
key   = mx.array([0.9, 0.4, -0.2, 0.7])   # "What do I contain?"

# Dot product = element-wise multiply, then sum
dot_manual = mx.sum(query * key)
dot_builtin = query @ key  # Same thing, cleaner syntax

mx.eval(dot_manual, dot_builtin)
print(f"Query:   {query.tolist()}")
print(f"Key:     {key.tolist()}")
print(f"Dot product (manual):  {dot_manual.item():.4f}")
print(f"Dot product (@ op):    {dot_builtin.item():.4f}")

# Cosine similarity (normalized dot product)
def cosine_similarity(a, b):
    return (a @ b) / (mx.sqrt(mx.sum(a**2)) * mx.sqrt(mx.sum(b**2)))

sim = cosine_similarity(query, key)
mx.eval(sim)
print(f"Cosine similarity:     {sim.item():.4f}")
print(f"\n💡 High dot product = vectors point in similar direction = high attention weight")

In [ ]:
# ============================================================
# 2.1.2 Matrix Multiplication — The Workhorse of Neural Nets
# ============================================================
# Every layer in a transformer: output = input @ weights + bias
# Understanding shapes is CRITICAL.

import mlx.core as mx

# Simulate a batch of token embeddings going through a linear layer
batch_size = 2
seq_len = 4
d_model = 8
d_ff = 32  # Feed-forward hidden dim (typically 4x d_model)

# Input: batch of sequences, each token is a d_model-dim vector
x = mx.random.normal((batch_size, seq_len, d_model))
print(f"Input shape:  {x.shape}")
print(f"  → {batch_size} sequences, {seq_len} tokens each, {d_model}-dim embeddings")

# Weight matrix (this is what the model LEARNS)
W = mx.random.normal((d_model, d_ff))
print(f"Weight shape: {W.shape}")
print(f"  → Projects {d_model}-dim → {d_ff}-dim")

# Forward pass: matrix multiply
output = x @ W
mx.eval(output)
print(f"Output shape: {output.shape}")
print(f"  → Same batch & seq_len, now {d_ff}-dim per token")

# Benchmark matmul on Metal GPU
import time

sizes = [512, 1024, 2048, 4096]
print(f"\nMatrix Multiply Benchmark (Metal GPU):")
print(f"{'Size':>6} {'Time (ms)':>10} {'GFLOPS':>10}")
print("-" * 30)

for n in sizes:
    a = mx.random.normal((n, n))
    b = mx.random.normal((n, n))
    mx.eval(a, b)  # Ensure data is ready
    
    # Warmup
    c = a @ b
    mx.eval(c)
    
    # Benchmark
    start = time.perf_counter()
    iters = 20
    for _ in range(iters):
        c = a @ b
        mx.eval(c)
    elapsed = (time.perf_counter() - start) / iters
    
    gflops = (2 * n**3) / (elapsed * 1e9)  # 2N^3 FLOPs for matmul
    print(f"{n:>6} {elapsed*1000:>9.2f}  {gflops:>9.1f}")

In [ ]:
# ============================================================
# 2.1.5 Cross-Entropy Loss — How We Train LLMs
# ============================================================
import mlx.core as mx
import numpy as np

# Cross-entropy: measures how different predicted distribution is from target.
# In LLM training: target = one-hot (the actual next token)

vocab_size = 10
target_token = 3  # The correct next token

# Good prediction (model is confident about token 3)
good_logits = mx.array([-1., -1., -1., 5., -1., -1., -1., -1., -1., -1.])

# Bad prediction (model is confused)
bad_logits = mx.array([1., 1., 1., 1.2, 1., 1., 1., 1., 1., 1.])

def cross_entropy_loss(logits, target_idx):
    """Cross-entropy loss for a single prediction."""
    log_probs = logits - mx.logsumexp(logits, axis=-1, keepdims=True)  # log-softmax
    return -log_probs[target_idx]

loss_good = cross_entropy_loss(good_logits, target_token)
loss_bad = cross_entropy_loss(bad_logits, target_token)

mx.eval(loss_good, loss_bad)

print(f"Good prediction loss:   {loss_good.item():.4f}")
print(f"Bad prediction loss:    {loss_bad.item():.4f}")
print(f"\nTheoretical minimum:    0.0 (perfect prediction)")
print(f"Theoretical max (uniform over {vocab_size}): {np.log(vocab_size):.4f}")
print(f"\nPerplexity = exp(loss):")
print(f"  Good: {np.exp(loss_good.item()):.2f} (model is almost certain)")
print(f"  Bad:  {np.exp(loss_bad.item()):.2f} (confused among ~{int(np.exp(loss_bad.item()))} tokens)")
print(f"\n💡 Perplexity is the 'effective vocabulary size' the model is confused about.")
print(f"   GPT-4 level perplexity on English text: ~5-10")
print(f"   Random guessing over 50K vocab: perplexity = 50,000")

In [ ]:
# ============================================================
# 3.3 Real-World Tokenizer (tiktoken — GPT-4's tokenizer)
# ============================================================
try:
    import tiktoken
    
    enc = tiktoken.get_encoding("cl100k_base")  # GPT-4 tokenizer
    
    texts = [
        "Hello, world!",
        "The quick brown fox jumps over the lazy dog.",
        "def fibonacci(n): return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
        "Apple Silicon M4 Pro with unified memory architecture.",
        "MLX is Apple's machine learning framework for Apple Silicon.",
    ]
    
    print("GPT-4 Tokenizer (cl100k_base, vocab=100K)")
    print("=" * 60)
    for text in texts:
        tokens = enc.encode(text)
        decoded_tokens = [enc.decode([t]) for t in tokens]
        words = text.split()
        print(f"\n\"{text}\"")
        print(f"  Tokens ({len(tokens)}): {decoded_tokens}")
        print(f"  IDs: {tokens}")
        print(f"  Fertility: {len(tokens)/len(words):.2f} tokens/word")

except ImportError:
    print("Install tiktoken: pip install tiktoken")

---
# ═══════════════════════════════════════════════════════════════
# PHASE 3: Tokenization
# ═══════════════════════════════════════════════════════════════

In [ ]:
# ============================================================
# 3.2 Byte-Pair Encoding (BPE) from Scratch
# ============================================================
# BPE iteratively merges the most frequent adjacent pair.
# This is the actual algorithm behind GPT tokenizers.

def train_bpe(text, num_merges=15):
    """Train a simple BPE tokenizer."""
    tokens = list(text)
    merges = {}  # (pair) -> merged_token
    
    print(f"Starting: {len(tokens)} tokens")
    
    for i in range(num_merges):
        # Count adjacent pairs
        pairs = {}
        for j in range(len(tokens) - 1):
            pair = (tokens[j], tokens[j+1])
            pairs[pair] = pairs.get(pair, 0) + 1
        
        if not pairs:
            break
        
        # Find most frequent pair
        best_pair = max(pairs, key=pairs.get)
        count = pairs[best_pair]
        new_token = best_pair[0] + best_pair[1]
        merges[best_pair] = new_token
        
        # Apply merge
        new_tokens = []
        j = 0
        while j < len(tokens):
            if j < len(tokens) - 1 and (tokens[j], tokens[j+1]) == best_pair:
                new_tokens.append(new_token)
                j += 2
            else:
                new_tokens.append(tokens[j])
                j += 1
        tokens = new_tokens
        
        print(f"  Merge {i+1:>2d}: '{best_pair[0]}' + '{best_pair[1]}' → '{new_token}' "
              f"(freq={count}, tokens now={len(tokens)})")
    
    return tokens, merges

corpus = "the cat sat on the mat. the cat ate the rat. the rat sat on the cat."
final_tokens, merges = train_bpe(corpus, num_merges=15)

print(f"\nFinal tokens ({len(final_tokens)}): {final_tokens}")
print(f"Compression: {len(corpus)} chars → {len(final_tokens)} tokens "
      f"({len(final_tokens)/len(corpus)*100:.1f}%)")

## Phase 3: Tokenization — How Text Becomes Numbers

LLMs don't see text. They see sequences of integers (token IDs). Tokenization is the bridge.

### Evolution:
1. **Character-level**: Each char = 1 token. Simple but sequences are very long.
2. **Word-level**: Each word = 1 token. Can't handle new/rare words.
3. **BPE (Byte-Pair Encoding)**: The sweet spot. Learns subword units from data.
4. **SentencePiece**: Language-agnostic BPE. Used by LLaMA, Gemma.
5. **Byte-level BPE**: Operates on raw bytes. Never has unknown tokens. Used by GPT-2/3/4.

### Key numbers:
- GPT-4: ~100K vocabulary (cl100k_base)
- LLaMA 3: 128K vocabulary
- Gemma: 256K vocabulary
- Average English word ≈ 1.3 tokens with modern tokenizers

In [ ]:
# ============================================================
# 3.1 Character-Level Tokenization (The Simplest)
# ============================================================
# This is where Karpathy's nanoGPT starts. Simple but inefficient.

text = "Hello, World! LLMs are fascinating."

# Build vocabulary from text
chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

print(f"Text: '{text}'")
print(f"Vocabulary size: {len(chars)}")
print(f"Characters: {chars}")

# Encode
encoded = [char_to_idx[c] for c in text]
print(f"\nEncoded ({len(encoded)} tokens): {encoded}")

# Decode
decoded = ''.join([idx_to_char[i] for i in encoded])
print(f"Decoded: '{decoded}'")

print(f"\n⚠️ Problem: {len(text)} characters = {len(encoded)} tokens")
print(f"   With BPE this would be ~8-10 tokens. 3-4x more efficient.")
print(f"   Longer sequences = quadratic attention cost = much slower.")